# CLT Inventory Pipeline

Run cells top-to-bottom. Each stage is idempotent — re-running a cell is safe.

**Prereqs:** `.env` with `DATABASE_URL` and `SERPAPI_KEY`; `pip install -r requirements.txt` in the activated venv.

## 0. Bootstrap schema (idempotent)

In [ ]:
from pipeline import db
db.bootstrap_schema()
print('schema ready')

## 1. Seed from existing clts.json

In [ ]:
from pipeline.config import SITE_CLTS_JSON
from pipeline.seeds import parse_existing_clts
from pipeline.stages import upsert_clts

rows = parse_existing_clts(SITE_CLTS_JSON)
upsert_clts(rows)
print(f'seeded {len(rows)} from existing clts.json')

## 2. Seed from directories

Skip the directory you couldn't capture during Task 14; the rest of the pipeline still works.

In [ ]:
from pathlib import Path
from pipeline.seeds import parse_grounded_solutions, parse_center_clt
from pipeline.stages import upsert_clts

for fixture, parser in [
    (Path('tests/fixtures/grounded_solutions.html'), parse_grounded_solutions),
    (Path('tests/fixtures/center_clt.html'), parse_center_clt),
]:
    if not fixture.exists():
        print(f'skipping {fixture.name} (no fixture)')
        continue
    rows = parser(fixture.read_text(encoding='utf-8'))
    upsert_clts(rows)
    print(f'{fixture.name}: seeded {len(rows)}')

## 3. State-by-state SerpAPI sweep

In [ ]:
from pipeline.config import load_env
from pipeline.serpapi import SerpApiClient, PostgresCache
from pipeline.seeds import US_STATES, build_state_sweep_query, parse_state_sweep_results
from pipeline.stages import upsert_clts

client = SerpApiClient(api_key=load_env()['SERPAPI_KEY'], cache=PostgresCache())
total = 0
for state in US_STATES:
    payload = client.search(build_state_sweep_query(state))
    rows = parse_state_sweep_results(payload, state=state)
    if rows:
        upsert_clts(rows)
        total += len(rows)
    print(f'{state}: {len(rows)} candidates')
print(f'total swept: {total}')

## 4. Discover URLs for CLTs without one

In [ ]:
from pipeline.stages import run_discover
print(run_discover())

## 5. Crawl

In [ ]:
from pipeline.stages import run_crawl
print(run_crawl())

## 6. Rescue dead/parked sites

In [ ]:
from pipeline.stages import run_rescue, run_crawl
print(run_rescue())
print(run_crawl())

## 7. Extract emails

In [ ]:
from pipeline.stages import run_extract
print(run_extract())

## 8. Diagnostics

In [ ]:
from pipeline.db import connect
with connect() as conn, conn.cursor() as cur:
    cur.execute('SELECT status, count(*) FROM inv_clts GROUP BY status ORDER BY 2 DESC')
    print('CLT status distribution:')
    for status, n in cur.fetchall():
        print(f'  {status:20s} {n}')
    cur.execute('SELECT error, count(*) FROM inv_pages WHERE error IS NOT NULL GROUP BY error ORDER BY 2 DESC LIMIT 10')
    print('\nTop crawl errors:')
    for err, n in cur.fetchall():
        print(f'  {n:5d} {err}')
    cur.execute('SELECT count(DISTINCT clt_id) FROM inv_emails')
    print(f'\nCLTs with at least one email: {cur.fetchone()[0]}')

## 9. Export to src/data/clts.json (opt-in)

This is the only cell that writes back into the site repo. Run it when you're satisfied with the inventory.

In [ ]:
from pipeline.export import export_to_clts_json
summary = export_to_clts_json(dry_run=True)
print('dry-run:', summary)
# When ready, change to dry_run=False:
# print(export_to_clts_json(dry_run=False))